In [ ]:
%load_ext autoreload
%autoreload 2


---

# FULL Ready to analyse 

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.append(os.path.abspath("../src"))

from dotenv import load_dotenv
load_dotenv()

import os
import json
import pandas as pd
import s3fs
import matplotlib.pyplot as plt

# ----------------------------------------------------------
# Imports from your utils (these already exist)
# ----------------------------------------------------------
from utils.day_loader import DayDataLoader
from utils.player_utils import PlayerNameMapper
from utils.match_loader import MatchesLoader
from utils.pitch_calibration import calibrate_pitch_from_df, attach_xy_from_pitch

# NEW: already implemented in your utils
from utils.match_phases import label_match_phases
from utils.player_status import label_active_players, detect_substitutions

# ----------------------------------------------------------
# S3 filesystem
# ----------------------------------------------------------
fs = s3fs.S3FileSystem(anon=False)
BUCKET = os.getenv("S3_BUCKET", "ucl-ai-soccormon-dataset")
print("S3 ready:", BUCKET)

# ----------------------------------------------------------
# Load matches + filter Rosenborg
# ----------------------------------------------------------
match_loader = MatchesLoader(r"D:\repos\UCLAI\data\kamper.xlsx")
matches = match_loader.load()

rosenborg = matches[
    matches["home"].str.contains("rosenborg", case=False, na=False)
    | matches["away"].str.contains("rosenborg", case=False, na=False)
].reset_index(drop=True)

display(rosenborg.head())


In [ ]:

# ----------------------------------------------------------
# Player mapping + day loader
# ----------------------------------------------------------
mapper = PlayerNameMapper("src/config/player_map.json")
day_loader = DayDataLoader(fs, mapper)


# ----------------------------------------------------------
# Select match + load raw GPS
# ----------------------------------------------------------
gnum = 8  # change freely
row = rosenborg.iloc[gnum - 1]

date_str = row["date"]  # YYYY-MM-DD
year = date_str[:4]
month = date_str[:7]

prefix = f"{BUCKET}/data/objective_TEAM_A_{year}/{month}/{date_str}"
print("Loading prefix:", prefix)

df_raw = day_loader.load_day(prefix)
print("Raw shape:", df_raw.shape)
display(df_raw.head())


In [ ]:

# ----------------------------------------------------------
# Downsample to 1 Hz
# ----------------------------------------------------------
df_1hz = day_loader.downsample_1hz(df_raw, method="first")
print("1 Hz shape:", df_1hz.shape)
df_1hz.head()

# ----------------------------------------------------------
# Pitch calibration
# ----------------------------------------------------------
with open("toppserien_pitches.json", "r", encoding="utf-8") as f:
    pitches = json.load(f)

stadium, center_latlon, R, pitch_xy = calibrate_pitch_from_df(
    df_1hz, pitches, lat_col="lat", lon_col="lon"
)

df_xy = attach_xy_from_pitch(
    df_1hz,
    center_latlon,
    R,
    lat_col="lat",
    lon_col="lon",
    stadium_name=stadium,
)

print("XY attached:", df_xy.shape)
df_xy.head()


In [ ]:

# ----------------------------------------------------------
# Label match phases (pre, 1H, HT, 2H, post)
# ----------------------------------------------------------
# MatchesLoader provides kickoff in local time already
kickoff_ts = pd.to_datetime(row["date"] + " " + row["time"])

df_xy = label_match_phases(df_xy, kickoff_ts)
df_xy["match_phase"].value_counts()

# Keep only match periods
df_match = df_xy[df_xy["match_phase"].isin(["1H", "2H"])].copy()
print("Match-only rows:", df_match.shape)

In [ ]:

# ----------------------------------------------------------
# Active/bench labelling
# ----------------------------------------------------------
df_labeled = label_active_players(
    df_match,
    pitch_xy,
    player_col="player_name",
    x_col="x_m",
    y_col="y_m",
    active_depth_m=3.0,
    activate_s=60,
    bench_off_s=120,
)

print("Active/bench labelling complete.")
df_labeled.head()


In [ ]:

# ----------------------------------------------------------
# Substitution detection
# ----------------------------------------------------------
subs = detect_substitutions(df_labeled)
print("Detected substitutions:")
display(subs)


In [ ]:
# ----------------------------------------------------------
# Animation with subs coloured differently
# ----------------------------------------------------------
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np

df_sorted = df_labeled.sort_values("timestamp")
times = df_sorted["timestamp"].drop_duplicates().tolist()
times = times[::5]  # fewer frames

fig, ax = plt.subplots(figsize=(10, 6))
draw_pitch(ax, pitch_xy, active_depth_m=6)

# We will update both coordinates AND colours
scat = ax.scatter([], [], s=70, edgecolors="k")

def init():
    scat.set_offsets(np.empty((0, 2)))
    scat.set_facecolors([])   # reset colours
    return scat,

def update(i):
    t = times[i]
    snap = df_sorted[df_sorted["timestamp"] == t]

    xy = np.c_[snap["x_m"], snap["y_m"]]

    # Colour by status
    # active → blue, subs (bench) → red
    colours = np.where(
        snap["player_status"].eq("active"),
        "dodgerblue",
        "crimson"
    )

    scat.set_offsets(xy)
    scat.set_facecolors(colours)

    ax.set_title(str(t))
    return scat,

anim = FuncAnimation(
    fig,
    update,
    init_func=init,
    frames=len(times),
    interval=120,
    blit=False
)

HTML(anim.to_jshtml())


## ANALYSIS 

In [ ]:
df_flock = df_labeled.query("match_phase in ['1H','2H'] and player_status == 'active'").copy()

print("Frames for flocking:", df_flock.shape)


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull

def compute_flocking_metrics(df_frame):
    """df_frame = all active players at a single timestamp."""
    X = df_frame["x_m"].to_numpy()
    Y = df_frame["y_m"].to_numpy()
    pts = np.column_stack([X, Y])

    # centroid
    centroid = pts.mean(axis=0)

    # team spread (mean distance to centroid)
    spread = np.linalg.norm(pts - centroid, axis=1).mean()

    # nearest-neighbour distance
    if len(pts) > 1:
        dmat = np.linalg.norm(pts[:,None,:] - pts[None,:,:], axis=2)
        np.fill_diagonal(dmat, np.inf)
        nnd = dmat.min(axis=1).mean()
    else:
        nnd = np.nan

    # convex hull area (team surface)
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        area = hull.area   # perimeter-like measure
        hull_area = hull.volume  # 2D area
    else:
        area = np.nan
        hull_area = np.nan

    return {
        "centroid_x": centroid[0],
        "centroid_y": centroid[1],
        "spread": spread,
        "nearest_neighbour": nnd,
        "hull_perimeter": area,
        "hull_area": hull_area,
    }


In [ ]:
flock_rows = []

for t, g in df_flock.groupby("timestamp"):
    metrics = compute_flocking_metrics(g)
    metrics["timestamp"] = t
    flock_rows.append(metrics)

df_flock_metrics = pd.DataFrame(flock_rows).sort_values("timestamp")

df_flock_metrics.head()


## Team spread over time

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,4))
plt.plot(df_flock_metrics["timestamp"], df_flock_metrics["spread"], lw=2)
plt.title("Team Spread Over Time")
plt.xlabel("Time")
plt.ylabel("Average Distance From Centroid (m)")
plt.grid(True)
plt.tight_layout()
plt.show()


## convex hull area

In [ ]:
plt.figure(figsize=(14,4))
plt.plot(df_flock_metrics["timestamp"], df_flock_metrics["hull_area"], color="forestgreen", lw=2)
plt.title("Team Convex Hull Area (Surface Occupied)")
plt.xlabel("Time")
plt.ylabel("Area (m²)")
plt.grid(True)
plt.tight_layout()
plt.show()


## Clean match only data 

In [ ]:
# Keep only 1H and 2H active players
df_match = df_labeled[df_labeled["match_phase"].isin(["1H", "2H"])]
df_match = df_match[df_match["player_status"] == "active"].copy()

df_match = df_match.sort_values("timestamp")
print(df_match.shape)


In [ ]:
import numpy as np

def compute_orientations(df):
    df = df.sort_values(["player_name", "timestamp"]).copy()

    # per-player finite difference velocities
    df["vx"] = df.groupby("player_name")["x_m"].diff()
    df["vy"] = df.groupby("player_name")["y_m"].diff()

    # orientation angle in radians
    df["theta"] = np.arctan2(df["vy"], df["vx"])

    return df

df_orient = compute_orientations(df_match)


In [ ]:
def compute_team_orientation(df):
    teams = []

    for t, g in df.groupby("timestamp"):
        vx = g["vx"].mean()
        vy = g["vy"].mean()
        theta_team = np.arctan2(vy, vx)
        teams.append({"timestamp": t, "theta_team": theta_team})

    return pd.DataFrame(teams)

df_team_orient = compute_team_orientation(df_orient)


In [ ]:
def compute_global_alignment(df_orient, df_team_orient):
    merged = df_orient.merge(df_team_orient, on="timestamp", how="left")

    rows = []
    for t, g in merged.groupby("timestamp"):
        dtheta = g["theta"] - g["theta_team"]
        order = np.abs(np.mean(np.exp(1j * dtheta)))
        rows.append({"timestamp": t, "alignment": order})

    return pd.DataFrame(rows)

df_align = compute_global_alignment(df_orient, df_team_orient)


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(df_align["timestamp"], df_align["alignment"], lw=2)
plt.ylim(0,1)
plt.title("Global Alignment Φ(t)")
plt.ylabel("Alignment (0–1)")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.neighbors import NearestNeighbors

def compute_local_alignment(df, k=3):
    rows = []

    for t, g in df.groupby("timestamp"):
        XY = g[["x_m", "y_m"]].values
        thetas = g["theta"].values

        nn = NearestNeighbors(n_neighbors=min(k+1, len(g))).fit(XY)
        dists, idxs = nn.kneighbors(XY)

        # each player ignores self (index 0)
        local_vals = []
        for i in range(len(g)):
            neigh = idxs[i][1:]      # k neighbours
            dtheta = thetas[neigh] - thetas[i]
            local_vals.append(np.abs(np.mean(np.exp(1j * dtheta))))

        rows.append({
            "timestamp": t,
            "local_alignment": np.mean(local_vals)
        })

    return pd.DataFrame(rows)

df_local = compute_local_alignment(df_orient)


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(df_local["timestamp"], df_local["local_alignment"], color="orange", lw=2)
plt.title("Local Alignment (k=3)")
plt.ylabel("Local Φ")
plt.grid(True)
plt.show()


In [ ]:
def compute_speed_synchrony(df):
    rows = []
    for t, g in df.groupby("timestamp"):
        v = np.sqrt(g["vx"]**2 + g["vy"]**2).values
        if len(v) < 3:
            continue
        sync = np.std(v) / (np.mean(v) + 1e-6)
        rows.append({"timestamp": t, "speed_synchrony": sync})
    return pd.DataFrame(rows)

df_sync = compute_speed_synchrony(df_orient)


In [ ]:
plt.figure(figsize=(14,4))
plt.plot(df_sync["timestamp"], df_sync["speed_synchrony"], color="green", lw=2)
plt.title("Speed Synchrony (σ/μ)")
plt.ylabel("Variability in Speed")
plt.grid(True)
plt.show()


In [ ]:
phi = df_align.set_index("timestamp")["alignment"]

high_phi = phi > 0.7
low_phi  = phi < 0.3

print("High φ periods:", high_phi.sum(), "frames")
print("Low φ periods:", low_phi.sum(), "frames")


In [ ]:
def compute_msd(df, mask):
    df2 = df[df["timestamp"].isin(mask.index[mask])].copy()
    df2["dx"] = df2.groupby("player_name")["x_m"].diff()
    df2["dy"] = df2.groupby("player_name")["y_m"].diff()
    return (df2["dx"]**2 + df2["dy"]**2).mean()

msd_high = compute_msd(df_match, high_phi)
msd_low  = compute_msd(df_match, low_phi)

print("MSD (high Φ):", msd_high)
print("MSD (low Φ):", msd_low)


Plot shows a smoothed order parameter Φ(t) with uncertainty. High Φ indicates coordinated group motion. Shaded halves allow testing if coherence systematically differs between 1H and 2H.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

phi = df_align.set_index("timestamp")["alignment"]
phi_smooth = pd.Series(gaussian_filter1d(phi.values, sigma=3), index=phi.index)

window = 15
roll = phi.rolling(window, center=True)
phi_lo = roll.quantile(0.1)
phi_hi = roll.quantile(0.9)

plt.figure(figsize=(15,5))
plt.plot(phi_smooth.index, phi_smooth, label="Smoothed Φ(t)", lw=2.5, color="black")

plt.fill_between(phi.index, phi_lo, phi_hi, color="gray", alpha=0.2,
                 label="10–90% envelope")

# highlight halves
for phase, color in [("1H", "#d0f0ff"), ("2H", "#ffd7d7")]:
    t_phase = df_match[df_match["match_phase"] == phase]["timestamp"]
    if len(t_phase) > 0:
        plt.axvspan(t_phase.min(), t_phase.max(), color=color, alpha=.18,
                    label=f"{phase} period")

plt.ylim(0,1)
plt.ylabel("Alignment Φ")
plt.title("Global Alignment Φ(t) — Smoothed\nHypothesis: coherence increases in attacking phases")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

x = df_local["timestamp"].astype("int64") // 10**9
y = df_local["local_alignment"]

sm = lowess(y, x, frac=0.1, it=0, return_sorted=False)

plt.figure(figsize=(15,5))
plt.plot(df_local["timestamp"], sm,
         lw=2.5, color="steelblue", label="LOWESS local alignment")
plt.ylim(0,1)
plt.grid(alpha=0.3)
plt.title("Local Alignment — Smooth Estimate\nHypothesis: compression → ↑ local alignment")
plt.ylabel("Local Φ")
plt.show()





In [ ]:
phi = df_align.set_index("timestamp")["alignment"]
high = phi > 0.7
low  = phi < 0.3

def collect_msds(df, mask):
    times = mask.index[mask]
    df2 = df[df["timestamp"].isin(times)]
    dx2 = (df2.groupby("player_name")["x_m"].diff())**2
    dy2 = (df2.groupby("player_name")["y_m"].diff())**2
    return (dx2 + dy2).dropna()

msd_high = collect_msds(df_match, high)
msd_low  = collect_msds(df_match, low)

plt.figure(figsize=(10,5))
plt.hist(msd_low, bins=40, density=True, alpha=0.5, label="Low Φ (disordered)",
         color="tomato")
plt.hist(msd_high, bins=40, density=True, alpha=0.5, label="High Φ (ordered)",
         color="seagreen")

plt.title("PDF of Step Sizes (MSDs)\nHypothesis: Disordered play yields larger diffusive steps")
plt.xlabel("Step² (m²)")
plt.ylabel("Density")
plt.legend()
plt.show()


In [ ]:
import numpy as np
import pandas as pd

df = df_match.sort_values(["player_name", "timestamp"]).copy()

# velocity components
df["vx"] = df.groupby("player_name")["x_m"].diff()
df["vy"] = df.groupby("player_name")["y_m"].diff()

# heading angle θ(t)
df["theta"] = np.arctan2(df["vy"], df["vx"])

# speed (optional)
df["speed"] = np.sqrt(df["vx"]**2 + df["vy"]**2)
def polarisation_func(angles):
    return np.abs(np.mean(np.exp(1j * angles)))

polarisation = df.groupby("timestamp")["theta"].apply(polarisation_func)


In [ ]:
import numpy as np
from scipy.spatial.distance import pdist

P = np.asarray(pitch_xy, dtype=float)   # ← convert list → array

pitch_diag = np.linalg.norm(P.max(axis=0) - P.min(axis=0))

def cohesion_func(sub):
    pts = sub[["x_m","y_m"]].values
    if len(pts) < 2:
        return np.nan
    d = pdist(pts).mean()
    return 1 - d / pitch_diag

cohesion = df.groupby("timestamp").apply(cohesion_func)


In [ ]:
def circular_entropy(angles):
    hist, _ = np.histogram(angles, bins=36, range=(-np.pi, np.pi), density=True)
    hist = hist[hist > 0]
    return -np.sum(hist * np.log(hist))

entropy = df.groupby("timestamp")["theta"].apply(circular_entropy)


In [ ]:
from scipy.ndimage import gaussian_filter1d

def smooth(s, sigma=3):
    return pd.Series(gaussian_filter1d(s.values, sigma=sigma), index=s.index)

Φ = polarisation
C = cohesion
H = entropy

Φs = smooth(Φ)
Cs = smooth(C)
Hs = smooth(H)

# normalise 0–1 for joint plotting
norm = lambda s: (s - s.min()) / (s.max() - s.min())

F = pd.DataFrame({
    "Φ_alignment": norm(Φs),
    "C_cohesion": norm(Cs),
    "H_entropy": norm(Hs)
})


In [ ]:
plt.figure(figsize=(16,6))

plt.plot(F.index, F["Φ_alignment"], lw=2.5, color="black",
         label="Polarisation Φ (alignment)")
plt.plot(F.index, F["C_cohesion"], lw=2.5, color="steelblue",
         label="Cohesion C (compactness)")
plt.plot(F.index, F["H_entropy"], lw=2.5, color="tomato",
         label="Heading Entropy H (disorder)")

plt.title("Collective Motion Order Parameters Over Match Time", fontsize=16)
plt.ylabel("Normalised Scale (0–1)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


---

# KEY FLOCKING ANALYSIS 1

In [ ]:
# Combine into a clean aligned frame
tmp = pd.DataFrame({"Φ": Φs, "C": Cs}).dropna()

Φ_clean = tmp["Φ"].values
C_clean = tmp["C"].values

# Scatter
plt.figure(figsize=(7,6))
plt.scatter(Φ_clean, C_clean, alpha=0.4, s=20, color="gray")

# Fit
m, b = np.polyfit(Φ_clean, C_clean, 1)
plt.plot([0,1], [m*0 + b, m*1 + b], color="red", lw=2.5)

plt.xlabel("Polarisation Φ")
plt.ylabel("Cohesion C")
plt.title("Φ vs C (Order–Cohesion Relationship)")
plt.grid(True)
plt.show()


In [ ]:
# --- Align Φs and Hs safely ---
tmp = pd.DataFrame({"Φ": Φs, "H": Hs}).dropna()

Φ_clean = tmp["Φ"].values
H_clean = tmp["H"].values

plt.figure(figsize=(7,6))

# Scatter
plt.scatter(Φ_clean, H_clean, alpha=0.35, s=22, color="gray")

# Regression line
m, b = np.polyfit(Φ_clean, H_clean, 1)
plt.plot([0,1], [m*0 + b, m*1 + b], color="red", lw=2.5)

plt.xlabel("Polarisation Φ", fontsize=12)
plt.ylabel("Heading Entropy H", fontsize=12)
plt.title("Alignment vs Angular Disorder", fontsize=14)
plt.grid(alpha=0.3)

plt.show()


The flocking analysis shows two distinct behaviours. First, the relationship between polarisation (Φ) and spatial cohesion (C) is weak and only mildly positive: the team remains tightly grouped even when their headings are disordered. This means cohesion is not an emergent flocking property but is instead imposed by tactical structure—players maintain formation and spacing independently of whether they are aligned in direction. Second, the relationship between polarisation and heading entropy (H) is strongly negative, showing classical flocking behaviour: when the team moves coherently, heading variability collapses, and when the system becomes disordered—during transitions, pressure moments, or structural resets—entropy rises sharply. Together, these results indicate that football teams behave like a tactically constrained flock: angular alignment and synchronisation follow flocking dynamics, while spatial cohesion is shaped by coaching and formation rather than emergent group mechanics.

---

## KEY AALYSIS 2 

In [ ]:
Phi = polarisation          # Φ(t)
C   = cohesion              # C(t)
H   = entropy    # H(t)

# and df_match: positions etc. with 'timestamp' and 'match_phase' in {1H,2H}

state = pd.DataFrame({
    "Φ": Phi,
    "C": C,
    "H": H,
}).sort_index()

# 10-second rolling mean + “confidence band” (±1 std)
win = "10s"
mean = state.rolling(win, min_periods=5).mean()
std  = state.rolling(win, min_periods=5).std()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

labels = {"Φ": "Polarisation Φ(t)", "C": "Cohesion C(t)", "H": "Heading entropy H(t)"}

for ax, col in zip(axes, ["Φ","C","H"]):
    ax.plot(state.index, state[col], alpha=0.25, lw=0.8, label=f"{col} raw")
    ax.plot(mean.index,  mean[col],  lw=2.0, label=f"{col} (10s mean)")
    ax.fill_between(
        mean.index,
        (mean[col] - std[col]),
        (mean[col] + std[col]),
        alpha=0.25,
        label="±1σ band"
    )
    ax.set_ylabel(labels[col])
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Time (timestamp)")
plt.tight_layout()
plt.show()


In [ ]:
# map match_phase onto state DF
phase_by_t = df_match.drop_duplicates("timestamp").set_index("timestamp")["match_phase"]
state["phase"] = phase_by_t.reindex(state.index).ffill()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for ax, col in zip(axes, ["Φ","C","H"]):
    for phase, color in zip(["1H","2H"], ["tab:blue","tab:orange"]):
        s = state[state["phase"] == phase]
        m = s[col].rolling("10s", min_periods=5).mean()
        ax.plot(m.index, m, label=f"{col} {phase}", lw=2, alpha=0.9,
                linestyle="-" if phase=="1H" else "--", color=color)
    ax.set_ylabel(col)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()


1. Order parameters Φ(t), C(t), H(t) with confidence bands
The smoothed trajectories show that polarisation Φ(t) fluctuates rapidly but tends to stabilise within characteristic “ordered islands”, where the team moves coherently in a shared direction. Heading entropy H(t) mirrors this almost perfectly: whenever Φ rises, H collapses, indicating low angular disorder; when Φ drops, H spikes. This confirms that directional coordination is the primary mode of collective order in football movement.

In contrast, cohesion C(t) remains high and comparatively stable throughout both halves. Its variance is small relative to Φ and H, indicating that spatial compactness is not an emergent flocking property but a tactical constraint: the team keeps its structure even when headings diverge. This reproduces a well-known distinction between animal flocks and human teams—humans maintain shape via instruction rather than self-organisation.


2. Phase-segmented dynamics (1H vs 2H)
Comparing 1H and 2H shows systematic behavioural differences:

2H exhibits lower entropy and slightly higher polarisation, suggesting the team moves more coherently in the second half—likely reflecting tactical adjustments (BEING A GOAL UP?) or opponent fatigue.

Cohesion is similar across halves, again supporting the idea that spacing is coached rather than emergent.

Transitions (tactical resets, turnovers, counter-press moments) are visible as sharp dips in Φ and spikes in H. These are the football equivalents of brief “disordered bursts” seen in biological collectives during reorganisation or threat response.

Together, this suggests that the team re-enters ordered states more quickly in 2H, a signature of improved synchronisation.

In [ ]:
# centroid per time
centroids = (df_match
             .groupby("timestamp")[["x_m","y_m"]]
             .mean()
             .rename(columns={"x_m":"cx","y_m":"cy"}))

df_c = (df_match
        .merge(centroids, on="timestamp", how="left")
        .assign(
            rx=lambda d: d["x_m"] - d["cx"],
            ry=lambda d: d["y_m"] - d["cy"],
        ))

# align Φ with positions
Phi_t = Phi.reindex(df_c["timestamp"]).to_numpy()
df_c["Phi"] = Phi.reindex(df_c["timestamp"]).values

phi_thr = 0.7
low_mask  = df_c["Phi"] < phi_thr
high_mask = df_c["Phi"] >= phi_thr

def compute_msd(df, max_lag_s=30):
    df = df.sort_values(["player_name","timestamp"])
    # relative position as complex
    z = df["rx"].to_numpy() + 1j*df["ry"].to_numpy()
    # we assume 1 Hz sampling
    msd = []
    lags = np.arange(1, max_lag_s+1)
    for tau in lags:
        dz = z[tau:] - z[:-tau]
        msd.append(np.mean(np.abs(dz)**2))
    return lags, np.array(msd)

lags_low,  msd_low  = compute_msd(df_c[low_mask])
lags_high, msd_high = compute_msd(df_c[high_mask])

plt.figure(figsize=(7,6))
plt.loglog(lags_low,  msd_low,  "o-", label="low Φ (disordered)")
plt.loglog(lags_high, msd_high, "o-", label="high Φ (ordered)")
plt.xlabel("Lag τ (s)")
plt.ylabel("MSD(τ)  [m²]")
plt.title("Centroid-frame MSD: ordered vs disordered phases")
plt.legend()
plt.grid(which="both", alpha=0.3)
plt.show()


. Conditional MSD: ordered vs disordered phases
When computing centroid-frame mean-squared displacement (MSD):

High-Φ (ordered) periods produce larger MSD at short times, meaning players move more efficiently relative to the team’s internal frame.

Low-Φ (disordered) periods show suppressed MSD at short times, suggesting internal turbulence: players counter-move, reorient, and re-space without producing coherent displacement.

Crucially, the MSD curves converge at longer lags, meaning long-timescale exploration of space is similar regardless of order state.
This mirrors collective motion in animals, where ordered phases support efficient transport, while disordered phases produce diffusive, corrective movement.

In [ ]:
# step lengths per player in centroid frame
df_c = df_c.sort_values(["player_name","timestamp"])
df_c["rx_next"] = df_c.groupby("player_name")["rx"].shift(-1)
df_c["ry_next"] = df_c.groupby("player_name")["ry"].shift(-1)

step = np.sqrt((df_c["rx_next"] - df_c["rx"])**2 +
               (df_c["ry_next"] - df_c["ry"])**2)

df_c["step"] = step

def estimate_levy_alpha(steps, s_min=1.0):
    s = steps[steps >= s_min].dropna()
    if len(s) < 20:
        return np.nan
    x = np.log(s)
    # MLE for continuous power-law tail p(s) ~ s^(-α)
    alpha = 1 + len(s) / ((x - x.min()).sum())
    return alpha

results = []
for phase in ["1H","2H"]:
    s_phase = df_c.loc[df_c["match_phase"] == phase, "step"]
    alpha = estimate_levy_alpha(s_phase, s_min=1.0)
    results.append({"phase": phase, "alpha": alpha})

levy_df = pd.DataFrame(results)
print(levy_df)


In [ ]:
plt.figure(figsize=(7,6))
for phase, color in [("1H","tab:blue"), ("2H","tab:orange")]:
    s = df_c.loc[df_c["match_phase"]==phase, "step"].dropna()
    s = s[s>=1.0]
    s_sorted = np.sort(s)
    ccdf = 1.0 - np.arange(1, len(s_sorted)+1)/ (len(s_sorted)+1)
    plt.loglog(s_sorted, ccdf, label=f"{phase}", alpha=0.8, color=color)

plt.xlabel("Step length s (m)")
plt.ylabel("P(S ≥ s)")
plt.title("Centroid-frame step-length tails by phase")
plt.legend()
plt.grid(which="both", alpha=0.3)
plt.show()


4. Step-length tails and Lévy behaviour (1H vs 2H)
The survival curves P(s ≥ S) show approximate power-law-like tails at intermediate scales (5–25m), a hallmark of Lévy-like intermittent bursts. These bursts correspond to:

recovery runs,

penetrative movements,

counter-attacks,

defensive sprints.

Comparing halves:

2H shows slightly heavier tails—more large steps—consistent with increased openness and transitional play in later phases of matches.

Neither half shows exponential truncation at short scales, suggesting movement intermittency persists throughout the match.

This supports the hypothesis that football player trajectories mix flocking-like order with Lévy-like burst dynamics, a combination also seen in:

foraging animals (albatrosses, spider monkeys),

bacterial collectives under chemotaxis,

shoaling fish transitioning between milling and schooling,

human hunter–gatherer search patterns,

neural population dynamics (high-order vs low-order firing regimes).

Summary (copy-friendly)

The results reveal a hybrid system:
directional synchrony (Φ, H) behaves like a classical flock, rising and falling as players align or disorder during play transitions.
But spatial cohesion (C) is maintained almost independently of alignment, indicating tactical enforcement rather than emergent self-organisation.

Ordered phases enable efficient, directed movement, while disordered phases exhibit internal turbulence and corrective motion, yet both contribute to Lévy-like intermittent burst dynamics in step-length distributions.

Together, these signatures place football teams within the family of structured collectives—systems that combine tactical or rule-based spacing with emergent alignment dynamics, much like:

hunter–gatherer tracking groups,

bacterial swarms that maintain density but vary directional order,

fish schools that hold shape while flexibly reorienting

## Part 5

In [ ]:
# --- 1. Compute centroid at each second ---
centroid = df_match.groupby("timestamp")[["x_m", "y_m"]].mean()

# --- 2. Mean distance of players to centroid at each time ---
def centroid_dist(sub):
    cx, cy = centroid.loc[sub.name]
    return np.sqrt((sub["x_m"] - cx)**2 + (sub["y_m"] - cy)**2).mean()

centroid_cohesion = df_match.groupby("timestamp").apply(centroid_dist)

# --- 3. Smooth for readability ---
cc_smooth = centroid_cohesion.rolling("10s", min_periods=5).mean()

# --- 4. Plot ---
plt.figure(figsize=(12,4))
plt.plot(centroid_cohesion.index, centroid_cohesion.values, alpha=0.3, label="raw")
plt.plot(cc_smooth.index, cc_smooth.values, lw=2, label="10s smoothed")
plt.title("Mean Distance to Team Centroid Over Time")
plt.ylabel("Mean radius (m)")
plt.xlabel("Time")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# --------------------------------------------------------
# Compute velocities per player (vx, vy) in m/s
# --------------------------------------------------------

df_vel = df_match.sort_values(["player_name", "timestamp"]).copy()

# time delta in seconds
df_vel["dt"] = df_vel.groupby("player_name")["timestamp"].diff().dt.total_seconds()

# position deltas
df_vel["dx"] = df_vel.groupby("player_name")["x_m"].diff()
df_vel["dy"] = df_vel.groupby("player_name")["y_m"].diff()

# velocity components
df_vel["vx"] = df_vel["dx"] / df_vel["dt"]
df_vel["vy"] = df_vel["dy"] / df_vel["dt"]

# drop first row per player (NaNs)
df_vel = df_vel.dropna(subset=["vx", "vy"]).copy()

# light smoothing to reduce GPS jitter (moving average 3 samples)
df_vel["vx"] = df_vel.groupby("player_name")["vx"].transform(lambda s: s.rolling(3, min_periods=1).mean())
df_vel["vy"] = df_vel.groupby("player_name")["vy"].transform(lambda s: s.rolling(3, min_periods=1).mean())


In [ ]:
from scipy.spatial.distance import pdist, squareform

corr_results = []

for t, frame in df_vel.groupby("timestamp"):
    if len(frame) < 3:
        continue

    # positions + velocities
    X = frame[["x_m", "y_m"]].values
    V = frame[["vx", "vy"]].values

    # pairwise distances
    D = squareform(pdist(X))

    # unit velocity vectors
    speeds = np.linalg.norm(V, axis=1, keepdims=True)
    speeds[speeds == 0] = 1e-6
    Vn = V / speeds

    # pairwise velocity alignment (cos θ_ij)
    dot = Vn @ Vn.T

    dij = D[np.triu_indices(len(frame), 1)]
    vij = dot[np.triu_indices(len(frame), 1)]

    corr_results.append(pd.DataFrame({
        "distance": dij,
        "vel_corr": vij,
        "timestamp": t
    }))

corr_df = pd.concat(corr_results, ignore_index=True)

# bin distances
bins = np.linspace(0, 60, 30)
corr_df["bin"] = pd.cut(corr_df["distance"], bins)
mean_corr = corr_df.groupby("bin")["vel_corr"].mean()
bin_centres = 0.5 * (bins[:-1] + bins[1:])

plt.figure(figsize=(8,5))
plt.plot(bin_centres, mean_corr.values, marker="o", lw=2)
plt.xlabel("Inter-player distance (m)")
plt.ylabel("Velocity correlation (cos θ)")
plt.title("Velocity–Velocity Correlation vs Distance")
plt.grid(True)
plt.show()
